# HiSparse 入门 (7/8)：后端接线 —— Core 与 Ascend 各管什么

> **一句话结论**：Core 持有生命周期与池分配口径，Ascend 后端包 `vllm_ascend/hisparse/` 供应算子、视图与传输适配；平台层通过注册把后端类挂进框架。

## 1. Core 与 Ascend 分工

| 职责 | 归属 |
| --- | --- |
| HMA 分配、Coordinator 准入/淘汰、请求生命周期 | vLLM Core |
| 分组预取（含 MTP 校验）、Host 镜像完成、draft finalizer | vLLM Core |
| 内存预算口径、`write_target()` 缓存与分组失效 ABI | vLLM Core |
| 驻留解析、Host 传输映射、共享 slab 视图适配 | Ascend 后端 |
| SFA 接入、流/事件围栏、算子 workspace 预分配 | Ascend 后端 |

## 2. 后端包模块一览

| 文件 | 职责 |
| --- | --- |
| `backend.py` | `AscendHiSparseBackend`：可用性检查、Host 注册、cache_ops 构造、D2H 分批（2047 上限）、profiling 前 workspace 预分配 |
| `binding.py` | slab 规格对齐、Indexer 视图、FP8 K·scale 别名 |
| `cache_ops.py` | `HiSparseNPUOps` + ResolverWorkspace：驻留解析、gather、失效、Resident 拷贝、行打包、散布 |
| `c8.py` | 打包、视图、物理索引映射、紧凑化 |
| `host_pool.py` | 锁页私有 Host + `aclrtHostRegister` |
| `host_mirror.py` | Target 写回与 draft 重叠围栏 |
| `recycled_invalidate.py` | 分组回收失效 |

## 3. 框架接线点

- `NPUPlatform.get_hisparse_backend_cls()` 返回 `vllm_ascend.hisparse.backend.AscendHiSparseBackend`；
- `worker.prepare_hisparse_workspaces` 在 profiling 前预分配 workspace；
- `patch/platform/patch_kv_cache_coordinator.py` 让 HiSparse 组走 Core 协调器；
- `attention/indexer.py` 提供 `AscendHiSparseIndexerBackend / MetadataBuilder`；
- `patch/worker/patch_attn_utils.py` 按 hisparse_config 选择 Indexer 后端。

## 4. 小结

- Core 管"什么时候留、什么时候淘汰、生命周期"；Ascend 管"怎么搬、怎么装、怎么接算子"；
- 后端包 7 个文件各管一块；平台注册 + worker 预分配 + coordinator 补丁完成接线；
- 最后一篇看 A5 的支持范围与当前状态。

**下一篇**：`08-scope-and-status-a5.ipynb`
